# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
from io import StringIO

import pandas as pd
import plotly.express as px

# Country, source and TWh come from the lecture's saved treemap leaves.
# Region comes from the saved World / Region / Country / Source hierarchy.
df = pd.read_csv(StringIO("""Country,Region,Source,TWh
Australia,Oceania,Coal,2919
Brazil,Latin America,Coal,312
Canada,North America,Coal,550
China,Asia,Coal,6929.1
France,Europe,Coal,120
Germany,Europe,Coal,537
India,Asia,Coal,4319
Japan,Asia,Coal,3347
Mexico,Latin America,Coal,949
South Africa,Africa,Coal,6727
South Korea,Asia,Coal,2706
United Kingdom,Europe,Coal,230
United States,North America,Coal,1015
Australia,Oceania,Hydro,448
Brazil,Latin America,Hydro,4857
Canada,North America,Hydro,4679
China,Asia,Hydro,488
France,Europe,Hydro,711
Germany,Europe,Hydro,453
India,Asia,Hydro,1156
Japan,Asia,Hydro,452
Mexico,Latin America,Hydro,791
Norway,Europe,Hydro,6174
South Africa,Africa,Hydro,403
South Korea,Asia,Hydro,114
United Kingdom,Europe,Hydro,174
United States,North America,Hydro,339
Australia,Oceania,Natural Gas,2599
Brazil,Latin America,Natural Gas,902
Canada,North America,Natural Gas,948
China,Asia,Natural Gas,872
France,Europe,Natural Gas,1160
Germany,Europe,Natural Gas,2293
India,Asia,Natural Gas,1018
Japan,Asia,Natural Gas,3130
Mexico,Latin America,Natural Gas,3137
Norway,Europe,Natural Gas,1193
Saudi Arabia,Middle East,Natural Gas,4573
South Africa,Africa,Natural Gas,472
South Korea,Asia,Natural Gas,3224
United Kingdom,Europe,Natural Gas,3139
United States,North America,Natural Gas,3083
Brazil,Latin America,Nuclear,118
Canada,North America,Nuclear,1290
China,Asia,Nuclear,394
France,Europe,Nuclear,6882
Germany,Europe,Nuclear,476
India,Asia,Nuclear,180
Japan,Asia,Nuclear,560
Mexico,Latin America,Nuclear,115
South Africa,Africa,Nuclear,323
South Korea,Asia,Nuclear,2370
United Kingdom,Europe,Nuclear,819
United States,North America,Nuclear,798
Australia,Oceania,Oil,2624
Brazil,Latin America,Oil,2973
Canada,North America,Oil,2315
China,Asia,Oil,1620
France,Europe,Oil,857
Germany,Europe,Oil,1361
India,Asia,Oil,2593
Japan,Asia,Oil,2965
Mexico,Latin America,Oil,4103
Norway,Europe,Oil,2228
Saudi Arabia,Middle East,Oil,5098
South Africa,Africa,Oil,1421
South Korea,Asia,Oil,2322
United Kingdom,Europe,Oil,2269
United States,North America,Oil,3220
Brazil,Latin America,Other Renewables,87
France,Europe,Other Renewables,413
Germany,Europe,Other Renewables,819
United Kingdom,Europe,Other Renewables,446
United States,North America,Other Renewables,229
Australia,Oceania,Solar,588
Brazil,Latin America,Solar,206
Canada,North America,Solar,193
China,Asia,Solar,231
France,Europe,Solar,376
Germany,Europe,Solar,814
India,Asia,Solar,245
Japan,Asia,Solar,173
Mexico,Latin America,Solar,205
Norway,Europe,Solar,228
South Africa,Africa,Solar,186
South Korea,Asia,Solar,213
United Kingdom,Europe,Solar,362
United States,North America,Solar,325
Australia,Oceania,Wind,989
Brazil,Latin America,Wind,419
Canada,North America,Wind,312
China,Asia,Wind,223
France,Europe,Wind,344
Germany,Europe,Wind,2348
India,Asia,Wind,422
Japan,Asia,Wind,275
Mexico,Latin America,Wind,188
Norway,Europe,Wind,260
Saudi Arabia,Middle East,Wind,96
South Africa,Africa,Wind,186
South Korea,Asia,Wind,165
United Kingdom,Europe,Wind,1775
United States,North America,Wind,413"""))
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable',
}
df['Source_Type'] = df['Source'].map(source_category)
assert len(df) == 103 and not df.duplicated(['Country', 'Source']).any()
assert df[['Region', 'Source_Type', 'TWh']].notna().all().all()
print(f"Loaded {len(df)} observations across {df['Country'].nunique()} countries.")


Loaded 103 observations across 15 countries.


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
df_fossil = df.loc[df['Source_Type'].eq('Fossil')].copy()
fossil_colors = {'Coal': '#0072B2', 'Oil': '#E69F00', 'Natural Gas': '#009E73'}
region_totals = df_fossil.groupby('Region')['TWh'].sum()
country_totals = df_fossil.groupby('Country')['TWh'].sum()

fig_fossil = px.treemap(
    df_fossil, path=['Region', 'Country', 'Source'], values='TWh',
    color='Source', color_discrete_map=fossil_colors,
    title=(f"{region_totals.idxmax()} leads in fossil energy use, with "
           f"{country_totals.idxmax()} the largest country in the dataset"),
)
leaf_colors = [
    fossil_colors[label] if node_id.count('/') == 2 else '#D9D9D9'
    for node_id, label in zip(fig_fossil.data[0].ids, fig_fossil.data[0].labels)
]
fig_fossil.update_traces(
    marker_colors=leaf_colors,
    texttemplate='%{label}<br>%{value:,.1f} TWh',
    hovertemplate='%{id}<br>%{value:,.1f} TWh<extra></extra>',
)
fig_fossil.update_layout(height=630)
fig_fossil.show()


Asia leads in fossil energy use, with Saudi Arabia the largest country in the dataset

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
low_carbon = (
    df.loc[df['Source_Type'].eq('Low-carbon')]
    .groupby('Country', as_index=False)['TWh'].sum()
    .sort_values('TWh', ascending=False)
    .assign(All='Low-carbon')
)
leader = low_carbon.iloc[0]
fig_low_tree = px.treemap(
    low_carbon, path=['All', 'Country'], values='TWh',
    title='Low-carbon energy by country (nuclear and hydro)',
)
fig_low_tree.update_traces(
    marker_colors=['#D9D9D9' if '/' not in node_id else '#0072B2'
                   for node_id in fig_low_tree.data[0].ids],
    texttemplate='%{label}<br>%{value:,.1f} TWh',
    hovertemplate='%{label}<br>%{value:,.1f} TWh<extra></extra>',
)
fig_low_tree.show()

fig_low_bar = px.bar(
    low_carbon, x='TWh', y='Country', orientation='h', text='TWh',
    color_discrete_sequence=['#0072B2'],
    title=f"{leader['Country']} leads in nuclear and hydro energy ({leader['TWh']:,.1f} TWh)",
)
fig_low_bar.update_traces(texttemplate='%{x:,.1f} TWh', textposition='outside', cliponaxis=False)
fig_low_bar.update_yaxes(categoryorder='array', categoryarray=low_carbon['Country'].tolist(),
                         autorange='reversed', title=None)
fig_low_bar.update_xaxes(title='Nuclear and hydro energy (TWh)', range=[0, leader['TWh'] * 1.18])
fig_low_bar.update_layout(height=560)
fig_low_bar.show()


Low-carbon energy by country (nuclear and hydro)

France leads in nuclear and hydro energy (7,593.0 TWh)

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The horizontal bar chart tells the ranking story more clearly. The countries are sorted, and each bar starts at zero, so readers can compare lengths and read the rank from top to bottom. The treemap shows how each country contributes to the total, but similar rectangle areas are harder to compare precisely.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
sources = list(fossil_colors)
fossil_by_region = (
    df_fossil.pivot_table(index='Region', columns='Source', values='TWh', aggfunc='sum')
    .reindex(columns=sources).fillna(0)
)
region_shares = fossil_by_region.div(fossil_by_region.sum(axis=1), axis=0)
assert region_shares.sum(axis=1).sub(1).abs().lt(1e-10).all()
coal_order = region_shares['Coal'].sort_values(ascending=False).index.tolist()
top_region = coal_order[0]
shares = region_shares.reset_index().melt(
    id_vars='Region', value_vars=sources, var_name='Source', value_name='Share',
)
fig_coal = px.bar(
    shares, x='Share', y='Region', color='Source', orientation='h', barmode='stack',
    category_orders={'Source': sources, 'Region': coal_order},
    color_discrete_map={'Coal': fossil_colors['Coal'], 'Oil': '#BDBDBD', 'Natural Gas': '#E0E0E0'},
    title=f"{top_region} has the largest coal share of fossil energy ({region_shares.loc[top_region, 'Coal']:.1%})",
)
for trace in fig_coal.data:
    trace.hovertemplate = '%{y}<br>%{x:.1%} of fossil TWh<extra>%{fullData.name}</extra>'
    if trace.name == 'Coal':
        trace.update(text=[f'{value:.1%}' for value in trace.x], texttemplate='%{x:.1%}', textposition='inside', textfont_color='white')
fig_coal.update_xaxes(tickformat='.0%', range=[0, 1], title='Share of regional fossil energy')
fig_coal.update_yaxes(autorange='reversed', title=None)
fig_coal.show()


Africa has the largest coal share of fossil energy (78.0%)

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The treemap answers **which country and fuel account for the largest absolute amount of fossil energy within a region**, because it shows country-level TWh and the hierarchy. The stacked bar answers **which region gets the largest share of its fossil energy from coal**, because every bar represents 100% of its own region’s fossil total and each coal segment starts at zero.